In [ ]:
import pandas as pd
data = pd.read_csv('./datas/naukri.csv', engine = 'python',on_bad_lines='skip')
df = pd.DataFrame(data)
df.head(1)

: 

In [138]:
df.shape

(23917, 7)

In [139]:
df.salary.unique()

array(['Not disclosed', '3-4.75 Lacs PA', '3-4.5 Lacs PA', ..., '49296.0',
       '40164.0', '73300.0'], dtype=object)

In [140]:
cols = ['company','salary','location']
for col in cols:
  df = df.drop(col , axis = 1)

In [141]:
df.head()

,title,experience,job-description,skills
0,Customer Engineer,2-7 Yrs,Provides prompt and courteous service response...,a+ technology recruitment customer engineering...
1,Service Operation Manager,4-8 Yrs,Managing a busy change team mailbox and ticket...,rackspace change management service operations...
2,Service Operation Manager,4-9 Yrs,Managing a busy change team mailbox and ticket...,operations management change management delive...
3,"Senior Manager, Business Operations",3-15 Yrs,Required bachelor s degree in data analytics o...,management senior management mis automation an...
4,Technical Support/ Customer Service Representa...,0-3 Yrs,Provides top quality customer service / techni...,sales process customer interaction inbound csr...


In [142]:
df.isna().sum()

,0
title,0
experience,0
job-description,0
skills,3902


In [143]:
df.dropna(inplace = True)

In [144]:
df.duplicated().sum()

np.int64(159)

In [145]:
df.drop_duplicates(inplace = True)


In [146]:
df.shape

(19856, 4)

In [147]:
df['Job_Text'] = df['title'] + " " + df['experience'] + " " +  df['skills'] + " " + df['job-description']

In [148]:
df.Job_Text

,Job_Text
0,Customer Engineer 2-7 Yrs a+ technology recrui...
1,Service Operation Manager 4-8 Yrs rackspace ch...
2,Service Operation Manager 4-9 Yrs operations m...
3,"Senior Manager, Business Operations 3-15 Yrs m..."
4,Technical Support/ Customer Service Representa...
...,...
21536,Social and Content Strategy Manager Mid-Senior...
21954,"Software Tester Entry level SYSTEM TESTING , M..."
22076,Financial Analyst Mid-Senior level EDUCATION A...
22227,Legal Data Analyst (Dallas/Chicago) Associate ...


In [149]:
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [150]:
from nltk.corpus import stopwords
stopwords = stopwords.words('english')
lemma = WordNetLemmatizer()
import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'http\S+|www\S+', ' ', text)

    text = re.sub(r'\S+@\S+', ' ', text)

    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    words = text.split()

    words = [
        lemma.lemmatize(word)
        for word in words
        if word not in stopwords
    ]

    return " ".join(words)

In [151]:
print(clean_text("Python https/www.com @$ Three % $ SQL Machine Learning"))

python three sql machine learning


In [152]:
x = df['Job_Text'].apply(clean_text)

In [153]:
x.head()

,Job_Text
0,customer engineer yr technology recruitment cu...
1,service operation manager yr rackspace change ...
2,service operation manager yr operation managem...
3,senior manager business operation yr managemen...
4,technical support customer service representat...


Tfidf vectorization


In [154]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,3),
)

In [155]:
job_vector = tfidf.fit_transform(x)

In [156]:
job_vector.shape

(19856, 10000)

In [157]:
from sklearn.metrics.pairwise import cosine_similarity

In [171]:
def recommender(resume,top_n):
  resume = clean_text(resume)
  resume_vector = tfidf.transform([resume])
  scores = cosine_similarity(resume_vector, job_vector)
  scores = scores.flatten()
  min_score = scores.min()
  max_score = scores.max()
  scaled_scores = (scores - min_score )/ (max_score - min_score)
  top_indices = scaled_scores.argsort()[::-1][:top_n]

  results = df.iloc[top_indices].copy()
  results["match_score"] = (scaled_scores[top_indices]*100).round(2)
  return results

In [178]:
resume = """ Seeking an Information Technology role where I can apply
my technical skills in software development, networking,
database management, and system administration.

Programming Languages:
Python, Java, C++, SQL

experience : 2years in full stack engineer, 5years in AI enginner

Tools:
Git, GitHub, VS Code"""

In [180]:
recommender(resume, 10)

,title,experience,job-description,skills,Job_Text,match_score
18558,React.js Full Stack Engineer,6-8 Yrs,Project Role : Full Stack Engineer . . Project...,Backend management Agile Interpersonal skills ...,React.js Full Stack Engineer 6-8 Yrs Backend m...,100.00
18774,"Full Stack Engineer Manager, Publisher Platform",14-15 Yrs,You should be comfortable in both front-end an...,software development life cycle software devel...,"Full Stack Engineer Manager, Publisher Platfor...",95.97
15187,Java Full Stack Development Full Stack Engineer,8-10 Yrs,"Technical Experience : A-Java 8, Git, Spring, ...",Web services Coding distribution system system...,Java Full Stack Development Full Stack Enginee...,86.53
15291,"Full Stack Engineer (Cassandra, Spark and Scala)",5-8 Yrs,We are looking for a mid-level full stack soft...,management backend python stack retail system ...,"Full Stack Engineer (Cassandra, Spark and Scal...",81.52
16696,"Full Stack Engineer, Jira Service Management",3-7 Yrs,"To help our teams work together effectively, t...",jira javascript git product management enginee...,"Full Stack Engineer, Jira Service Management 3...",79.51
14830,Software Developer,0-1 Yrs,Skills on Java /Fullstack / Desktop support /p...,JAVA/ PYTHON software development development ...,Software Developer 0-1 Yrs JAVA/ PYTHON softwa...,76.41
19771,Cloud Full Stack Developer,4-8 Yrs,Required Technical and Professional Expertise ...,Software development life cycle software devel...,Cloud Full Stack Developer 4-8 Yrs Software de...,73.65
19380,Full Stack Developer,4-6 Yrs,Your Role and ResponsibilitiesSoftware Enginee...,Java javas JSP Python SQL react.js full stack ...,Full Stack Developer 4-6 Yrs Java javas JSP Py...,68.92
15782,Principal Engineer (Full Stack),6-11 Yrs,You are an engineer at heart with a track reco...,budgeting jira technical backend operational e...,Principal Engineer (Full Stack) 6-11 Yrs budge...,67.70
15300,Lead Full-Stack Engineer,6-11 Yrs,The candidate should have experience building ...,management billing agile development backend c...,Lead Full-Stack Engineer 6-11 Yrs management b...,65.76


In [181]:
from google.colab import files

In [183]:
import joblib

In [185]:
x.to_csv('naukri_cleaned_data.csv',index = False)

In [186]:
files.download('naukri_cleaned_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>